# 03 — ARIMA Modelling & Anomaly Detection

Walk-forward ARIMA forecasting + z-score flagging to detect revenue anomalies.

**Method:**
1. Fit `ARIMA(2,1,2)` on rolling 7-day (168h) window
2. Forecast 1 step ahead with 95% confidence interval
3. Flag if actual value falls outside CI → **ARIMA anomaly**
4. Flag if |z-score| > 2.5 → **z-score anomaly**
5. Union of both flags = final anomaly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

ARIMA_ORDER = (2, 1, 2)
TRAIN_WINDOW = 168   # 7 days × 24h
Z_THRESH = 2.5

In [ ]:
df = pd.read_csv('../data/processed/ad_metrics_clean.csv', parse_dates=['datetime'])
print(f'Shape: {df.shape}')

## Single-metric ARIMA demo (eCPM)

We show a small window to illustrate the method before running the full pipeline.

In [ ]:
# Demo on first 300 data points
demo = df.head(300).copy()
series = demo['ecpm'].values
times = demo['datetime']

forecasts, lower_ci, upper_ci = [np.nan]*len(series), [np.nan]*len(series), [np.nan]*len(series)

for i in range(TRAIN_WINDOW, len(series)):
    train = series[i - TRAIN_WINDOW : i]
    try:
        model = ARIMA(train, order=ARIMA_ORDER)
        fit = model.fit()
        fc = fit.get_forecast(steps=1)
        forecasts[i] = fc.predicted_mean[0]
        ci = fc.conf_int(alpha=0.05)
        lower_ci[i] = ci.iloc[0, 0]
        upper_ci[i] = ci.iloc[0, 1]
    except:
        forecasts[i] = series[i-1]
        lower_ci[i] = series[i-1] - series[i - TRAIN_WINDOW:i].std() * 2
        upper_ci[i] = series[i-1] + series[i - TRAIN_WINDOW:i].std() * 2

forecasts = np.array(forecasts)
lower_ci = np.array(lower_ci)
upper_ci = np.array(upper_ci)

# Z-score flags
roll_mean = pd.Series(series).rolling(TRAIN_WINDOW, min_periods=1).mean().values
roll_std = pd.Series(series).rolling(TRAIN_WINDOW, min_periods=1).std().fillna(0.001).values
z_scores = (series - roll_mean) / roll_std
z_anom = np.abs(z_scores) > Z_THRESH
arima_anom = (series < lower_ci) | (series > upper_ci)
combined = z_anom | arima_anom

print(f'ARIMA anomalies: {arima_anom.sum()}')
print(f'Z-score anomalies: {z_anom.sum()}')
print(f'Combined (union): {combined.sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(times, series, color='#555', linewidth=0.9, label='Observed eCPM', zorder=2)

mask = ~np.isnan(forecasts)
ax.plot(times[mask], forecasts[mask], color='#2196F3', linewidth=0.9, linestyle=':', label='ARIMA Forecast', zorder=3)
ax.fill_between(times[mask], lower_ci[mask], upper_ci[mask], color='#2196F3', alpha=0.12, label='95% CI')

ax.scatter(times[combined], series[combined], color='#d62728', s=60, zorder=5,
           label='Anomaly', edgecolors='white', linewidths=0.5)

ax.set_title('eCPM — ARIMA Anomaly Detection (Demo Window)', fontsize=13, fontweight='bold')
ax.set_ylabel('eCPM ($)')
ax.set_xlabel('Datetime')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %Hh'))
plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## AIC-based ARIMA order selection

In [ ]:
# Quick AIC grid search on eCPM to validate (2,1,2)
train_sample = df['ecpm'].iloc[:200].values
results = []
for p in range(0, 4):
    for q in range(0, 4):
        try:
            m = ARIMA(train_sample, order=(p, 1, q)).fit()
            results.append({'p': p, 'q': q, 'AIC': round(m.aic, 2), 'BIC': round(m.bic, 2)})
        except:
            pass

res_df = pd.DataFrame(results).sort_values('AIC')
print('Top ARIMA orders by AIC:')
print(res_df.head(8).to_string(index=False))